# Clase 031 — Series de tiempo

**Parte 0** · VanderPlas cap. 3 § 3.12.

> 🎯 Parsear, indexar, resamplear y rolling sobre datos temporales.

> ⏱️ ~90 min

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
fechas = pd.date_range('2024-01-01', '2025-12-31', freq='D')
ventas = pd.Series(
    rng.normal(1000, 200, len(fechas)).cumsum().clip(min=0).astype(int),
    index=fechas,
    name='ventas',
)
print(ventas.head())

## 1️⃣ `pd.to_datetime` — parseo robusto

```python
pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')
```

`errors='coerce'` convierte lo no parseable a `NaT` (Not a Time) en vez de lanzar excepción.

In [ ]:
raw = pd.Series(['2024-01-15', '15/02/2024', '2024-03-20', 'foo', '2024-04-10'])
fechas_parsed = pd.to_datetime(raw, format='mixed', errors='coerce')
print(fechas_parsed)
print(f'\nNaT count: {fechas_parsed.isna().sum()}')

## 2️⃣ DatetimeIndex y slicing

Con índice datetime, puedes slicear con strings:

In [ ]:
# Trimestre Q1 2024
q1 = ventas.loc['2024-01':'2024-03']
print(f'Q1 2024: {len(q1)} días')

# Solo enero 2025
ene_25 = ventas.loc['2025-01']
print(f'Enero 2025: {len(ene_25)} días')

# Componentes
print(f'\naño/mes/día/dow de la primera fecha:')
print(f'  year={ventas.index[0].year}')
print(f'  month={ventas.index[0].month}')
print(f'  day_name={ventas.index[0].day_name()}')

## 3️⃣ Resampling — cambiar frecuencia

| Alias | Frecuencia |
|---|---|
| `'D'` | día |
| `'W'` | semana |
| `'ME'` | mes (end) |
| `'QE'` | trimestre |
| `'YE'` | año |
| `'h'` | hora |
| `'min'` | minuto |

Resampling **siempre requiere agregación**: sum, mean, last, ohlc.

In [ ]:
# Diaria → mensual
mensual = ventas.resample('ME').agg(['sum', 'mean', 'std']).round(0)
print(mensual.head())

# Diaria → semanal (suma)
semanal = ventas.resample('W').sum()
print(f'\nsemanas: {len(semanal)}')

## 4️⃣ Rolling windows — medias móviles

Ventana móvil: a cada punto, aplicar función a los últimos N puntos. Suaviza tendencias.

In [ ]:
rolling_7  = ventas.rolling(7).mean()
rolling_30 = ventas.rolling(30).mean()

fig, ax = plt.subplots(figsize=(11, 4))
ventas.plot(ax=ax, alpha=0.4, label='diaria', linewidth=0.7)
rolling_7.plot(ax=ax, label='rolling 7d', linewidth=1.2)
rolling_30.plot(ax=ax, label='rolling 30d', linewidth=1.5)
ax.set_title('Ventas — original vs ventanas móviles')
ax.legend()
ax.set_ylabel('ventas')
plt.tight_layout()
plt.show()

## 5️⃣ `shift` y `diff` — lag y variación

```python
s.shift(1)        # adelanta 1 paso (NaN al inicio)
s.diff(1)         # s - s.shift(1) → cambio absoluto
s.pct_change()    # cambio relativo (%)
```

In [ ]:
df = pd.DataFrame({
    'ventas'      : ventas,
    'lag_1'       : ventas.shift(1),
    'diff_1'      : ventas.diff(1),
    'pct_change'  : ventas.pct_change() * 100,
})
print(df.head(6).round(2))

## 6️⃣ Timezones — `tz_localize` y `tz_convert`

```python
s.tz_localize('UTC')           # asigna TZ (no convierte)
s.tz_convert('America/Santiago')  # convierte a otra TZ
```

Regla: primero **localize** (asigna), luego **convert** (mueve).

In [ ]:
naive = pd.Series([1, 2, 3], index=pd.date_range('2024-01-01', periods=3, freq='h'))
utc = naive.tz_localize('UTC')
scl = utc.tz_convert('America/Santiago')   # -3h o -4h según DST
print('UTC:'); print(utc)
print('Santiago:'); print(scl)

## ✅ Checklist

- [ ] Parseo fechas con `to_datetime(errors='coerce')`
- [ ] Indexo por fecha y sliceo con strings
- [ ] Resampleo a la frecuencia objetivo + agg
- [ ] Uso rolling para suavizar tendencias
- [ ] Sé hacer lag features con `shift`/`diff`

## 📝 Homework

Ver `README.md`. Parseo, slice por trimestre, resample mensual, rolling 7/30 con plot, diff.

## 📖 Definiciones y características

**`Timestamp` / `DatetimeIndex`**

Tipo nativo de pandas para fechas-horas. Vectorizado. Permite indexar con strings: `df.loc['2024-01':'2024-03']`.

**`pd.to_datetime`**

Conversor robusto string→Timestamp. `errors='coerce'` convierte fallos a `NaT` (Not a Time) en vez de excepción.

**Resampling**

Cambiar la frecuencia de una serie: diaria→mensual, horaria→semanal. Requiere **agregación** (sum, mean, last, ohlc). Códigos: `'D'`, `'W'`, `'ME'` (month-end), `'h'`, `'min'`.

**Rolling window**

Ventana móvil: para cada punto, aplicar función a los últimos N (`.rolling(N).mean()`). Suaviza tendencias, calcula medias móviles.

**`shift` / `diff` / `pct_change`**

**`shift(N)`**: desplaza N posiciones (NaN al inicio). **`diff(N)`**: `s - s.shift(N)` (cambio absoluto). **`pct_change()`**: cambio relativo.

**`tz_localize` vs `tz_convert`**

**`localize`** asigna TZ a fecha naive (no convierte). **`convert`** convierte de una TZ a otra (mantiene el instante). Patrón: localize primero, convert después.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `pd.to_datetime` falla con `ValueError` | Formato no estándar. **Fix**: `format='%d/%m/%Y'` explícito, o `errors='coerce'` para convertir fallos a NaT y diagnosticar después. |
| `df.loc['2024-13']` lanza KeyError | Mes inválido. **Fix**: verifica formato — pandas espera ISO (`'2024-01'`). |
| `resample('M')` warning sobre deprecation | Pandas 2.2+ prefiere `'ME'` (month-end) o `'MS'` (month-start) sobre `'M'` ambiguo. **Fix**: actualiza alias. |
| Resta de fechas da `Timedelta` en lugar de número | Es correcto — usa `.dt.days`, `.dt.seconds` para obtener escalar. `(d1 - d2).dt.days`. |
| Operaciones con TZ mezcladas: `Cannot compare tz-naive and tz-aware` | Una fecha tiene timezone, la otra no. **Fix**: alinea — `tz_localize('UTC')` a la naive. |

## ❓ Preguntas frecuentes

**❓ ¿Cuándo `DatetimeIndex`?**

Cuando vas a hacer `resample`, slicing por fechas, o cualquier operación temporal. `set_index('fecha')` después de parsear.

**❓ ¿`resample` o `groupby(date.dt.month)`?**

**`resample`** es nativo y maneja gaps + zonas horarias mejor. **`groupby`** para agrupaciones no temporales ("por mes ignorando año").

**❓ ¿Desde cuándo empieza a dar valores el rolling?**

Default: NaN en los primeros N-1 (no hay datos suficientes para la ventana). Si quieres usar lo que haya: `.rolling(N, min_periods=1)`.

**❓ ¿Cómo manejo zonas horarias en producción?**

**Regla**: guarda todo en UTC, convierte solo para mostrar. `df['fecha'] = pd.to_datetime(...).dt.tz_localize('UTC')` al ingerir, `tz_convert('America/Santiago')` al exhibir.

**❓ ¿`pd.date_range` o `pd.timestamp_range`?**

**`date_range`** — `timestamp_range` no existe. `pd.date_range('2024-01-01', '2024-12-31', freq='D')` para 365 fechas diarias.

## 🔗 Referencias

- VanderPlas cap. 3 § 3.12
- [pandas Time Series](https://pandas.pydata.org/docs/user_guide/timeseries.html)

➡️ **Siguiente:** [032 — eval y query](../032-pandas-eval-y-query/README.md)

## ✅ Soluciones de los ejercicios

A continuación, cada ejercicio de la sección `🧪 Ejercicios` del README resuelto y comentado. Todo el código es **ejecutable sin conexión** (datos sintéticos) e incluye `assert`/`print` para que compruebes el resultado. Intenta resolverlos por tu cuenta antes de mirar la solución.

**Ej. 1 — Parseo robusto** de fechas mixtas (`errors='coerce'`).

In [ ]:
import pandas as pd, numpy as np
fechas = pd.to_datetime(['2024-01-15', '15/02/2024', 'foo'],
                        dayfirst=True, format='mixed', errors='coerce')
print(fechas)
assert pd.isna(fechas[-1]) and fechas[:-1].notna().all()   # solo 'foo' -> NaT

**Ej. 2 — Slice por fecha** (Q1 2024).

In [ ]:
idx = pd.date_range('2024-01-01', periods=365, freq='D')
ventas = pd.Series(np.arange(365), index=idx)
q1 = ventas.loc['2024-01':'2024-03']
print('dias en Q1:', q1.shape[0])
assert q1.shape[0] == 91          # ene 31 + feb 29 (bisiesto) + mar 31

**Ej. 3 — Resample diario -> mensual** (suma).

In [ ]:
mensual = ventas.resample('ME').sum()     # 'ME' = month-end
print(mensual.head())
assert mensual.shape[0] == 12

**Ej. 4 — Rolling 7-day mean** (+ plot).

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
rng = np.random.default_rng(31)
serie = pd.Series(rng.normal(100, 10, 90), index=pd.date_range('2024-01-01', periods=90))
media7 = serie.rolling(7).mean()
fig, ax = plt.subplots()
ax.plot(serie.index, serie, alpha=.4, label='diario')
ax.plot(media7.index, media7, label='media 7d')
ax.legend(); plt.close(fig)
assert media7.iloc[:6].isna().all() and media7.iloc[6:].notna().all()
print('rolling(7): los primeros 6 valores son NaN (ventana incompleta).')

**Ej. 5 — `shift(1)`** para una lag feature.

In [ ]:
ventas_lag = ventas.shift(1)
assert pd.isna(ventas_lag.iloc[0]) and ventas_lag.iloc[1] == ventas.iloc[0]
print('ventas_lag_1 trae el valor del dia anterior. Primeros:\n', ventas_lag.head())